## Environment Setup

In [ ]:
!pip install -U torch torchvision torchaudio
!pip install -U transformers accelerate peft bitsandbytes datasets qwen-vl-utils sentence-transformers json-repair

## Imports & Global Configurations

In [1]:
import os
import re
import json
import copy
import torch
import numpy as np
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import PeftModel
from qwen_vl_utils import process_vision_info
from sentence_transformers import SentenceTransformer
import json_repair

# Force Hugging Face to download massive weights to the persistent volume
os.environ["HF_HOME"] = "/workspace/huggingface_cache"

# --- Global Configurations ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAFE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32

QWEN_BASE_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
EXTRACTOR_LORA_PATH = "/workspace/ImageToSpec_Stage1/qwen_lora_final"
VERIFIER_LORA_PATH = "/workspace/Stage2_ChartCheck/Stage2_ChartCheck/qwen_verifier_lora/checkpoint-900"

## Load Models (Multi-Adapter Setup)

In [2]:
print("1. Loading Qwen Base Model & Processor...")
qwen_processor = AutoProcessor.from_pretrained(QWEN_BASE_ID)
# Load directly into the variable name you want to use
qwen_model = AutoModelForImageTextToText.from_pretrained(
    QWEN_BASE_ID, 
    device_map="auto", 
    torch_dtype=SAFE_DTYPE, 
    attn_implementation="sdpa"
)

1. Loading Qwen Base Model & Processor...


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

## Data Pruning & Retrieval Logic

In [3]:
def retrieve_top_k_series(claim, spec, k=3):
    series_list = spec.get("ser", [])
    if not isinstance(series_list, list) or len(series_list) <= k: return spec
    docs = [claim] + [json.dumps(s) for s in series_list]
    with torch.no_grad():
        embs = semantic_retriever.encode(docs, convert_to_tensor=True)
        sims = torch.nn.functional.cosine_similarity(embs[0:1], embs[1:])
    top_k = sims.topk(k).indices.sort().values.tolist()
    spec["ser"] = [series_list[i] for i in top_k]
    spec["is_truncated"] = True
    return spec

def lightweight_prune_for_llm(raw_spec_dict, claim_text):
    """Surgically prunes the spec to preserve semantics and math, while dropping visual noise."""
    if not isinstance(raw_spec_dict, dict): return {}
    cleaned = copy.deepcopy(raw_spec_dict)
    
    if 'ser' not in cleaned or not isinstance(cleaned['ser'], list) or len(cleaned['ser']) == 0: 
        return cleaned

    def compress_val(v):
        if isinstance(v, float): return int(v) if v.is_integer() else round(v, 3)
        return v

    # 1. Strip Root-Level Visual/System Noise
    cleaned.pop('legend', None)
    cleaned.pop('tooltip', None)

    # 2. Apply Semantic Retriever 
    cleaned = retrieve_top_k_series(claim_text, cleaned, k=3)

    # 3. Clean Remaining Series
    topo = cleaned.get('topo', {})
    topo_type = str(topo.get('type', '')).strip().lower() if isinstance(topo, dict) else ''
    
    for ser in cleaned.get('ser', []):
        if isinstance(ser, dict):
            # Aggressively drop visual flags
            for key in ['ds', 'is_subsampled', 'critical_points_retained', 'color', 'uncertainty_metric']:
                ser.pop(key, None)
            
            # Pie charts do not mathematically possess trends
            if 'pie' in topo_type: 
                ser.pop('trend', None)
                ser.pop('stats', None)
            
            # Compress float values in data arrays
            if 'data' in ser and isinstance(ser['data'], list):
                compressed_data = []
                for pt in ser['data']:
                    if isinstance(pt, list):
                        compressed_data.append([compress_val(v) for v in pt])
                    else:
                        compressed_data.append(compress_val(pt))
                ser['data'] = compressed_data

    return cleaned

## Simulation Samples

In [4]:
simulation_samples = [
    # {
    #     "image_path": "17979.jpeg",
    #     "title_description": "2020s: The Billionaires' Decade? Cumulative wealth and number of billionaires worldwide",
    #     "chart_type": "bar_chart",
    #     "claim": "The cumulative wealth of billionaires steadily decreased between 2020 and 2026.",
    #     "truth" : " (REFUTED claim)"
    # },
    # {
    #     "image_path": "17979.jpeg",
    #     "title_description": "2020s: The Billionaires' Decade? Cumulative wealth and number of billionaires worldwide",
    #     "chart_type": "bar_chart",
    #     "claim": "The number of billionaires worldwide more than doubled from 2013 to 2026.",
    #     "truth" : " (SUPPORTED claim)"
    # },
    # {
    #     "image_path": "35976.jpeg",
    #     "title_description": "Content Marketers Embrace AI in Content Creation",
    #     "chart_type": "horizontal_bar_chart",
    #     "claim": "Process automation is the most common use case for AI among content marketers, utilized by over 70% of respondents.",
    #     "truth": " (REFUTED claim)"
    # },
    # {
    #     "image_path": "35976.jpeg",
    #     "title_description": "Content Marketers Embrace AI in Content Creation",
    #     "chart_type": "horizontal_bar_chart",
    #     "claim": "Over half of the surveyed content marketing professionals use AI tools for content creation, making it the top use case.",
    #     "truth" : " (SUPPORTED claim)"
    # },
    # {
    #     "image_path": "35957.jpeg",
    #     "title_description": "Web Censorship Cases Rebound in 2025",
    #     "chart_type": "clustered_bar_chart",
    #     "claim": "The number of new web censorship cases steadily decreased every year from 2022 to 2025.",
    #     "truth": " (REFUTED claim)"
    # },
    # {
    #     "image_path": "35957.jpeg",
    #     "title_description": "Web Censorship Cases Rebound in 2025",
    #     "chart_type": "clustered_bar_chart",
    #     "claim": "There were more new web censorship cases reported in 2025 than in any other year shown on the chart.",
    #     "truth" : " (SUPPORTED claim)"
    # }
    #https://docs.devexpress.com/WindowsForms/401311/controls-and-libraries/chart-control/series-views/2d-series-views/box-plot-chart
    {
        "image_path": "box-plot-preview.png",
        "title_description": "Experiment History: Comparison of Data Set 1 and Data Set 2 across multiple experiments",
        "chart_type": "box_plot",
        "claim": "In Experiment 18, Data Set 2 has a higher median value than Data Set 1.",
        "truth": "(SUPPORTED claim)"
    },
    {
        "image_path": "box-plot-preview.png",
        "title_description": "Experiment History: Comparison of Data Set 1 and Data Set 2 across multiple experiments",
        "chart_type": "box_plot",
        "claim": "Across most experiments shown, (blue) consistently displays a larger number of lower-end outliers compared to Data Set 2.",
        "truth": "(SUPPORTED claim)"
    },
    {
        "image_path": "box-plot-preview.png",
        "title_description": "Experiment History: Comparison of Data Set 1 and Data Set 2 across multiple experiments",
        "chart_type": "box_plot",
        "claim": "In Experiment 24, Data Set 2 shows a significantly higher maximum value, including outliers, than Data Set 1.",
        "truth": "(REFUTED claim)"
    },
    {
        "image_path": "box-plot-preview.png",
        "title_description": "Experiment History: Comparison of Data Set 1 and Data Set 2 across multiple experiments",
        "chart_type": "box_plot",
        "claim": "The interquartile range for Data Set 1 remains perfectly constant across all experiments from 18 to 25.",
        "truth": "(REFUTED claim)"
    },
    # https://jokergoo.github.io/ComplexHeatmap-reference/book/other-high-level-plots.html
    {
        "image_path": "heatmap.png",
        "title_description": "Density heatmap of variable m across categories C1 to C20 with percentile overlays",
        "chart_type": "density_heatmap",
        "claim": "There is a distinct shift in the data distribution starting at C11, where the highest density areas (dark red) move from being centered near 0 to being more spread out or slightly higher.",
        "truth" : "(SUPPORTED claim)"
    },
    {
        "image_path": "heatmap.png",
        "title_description": "Density heatmap of variable m across categories C1 to C20 with percentile overlays",
        "chart_type": "density_heatmap",
        "claim": "For category C8, the mean value is slightly higher than the 50% percentile (median) value.",
        "truth" : "(SUPPORTED claim)"
    },
    {
        "image_path": "heatmap.png",
        "title_description": "Density heatmap of variable m across categories C1 to C20 with percentile overlays",
        "chart_type": "density_heatmap",
        "claim": "The 100% percentile line (the top dashed line) remains at a constant value of m = 3.0 for every category from C1 to C20.",
        "truth" : "(REFUTED claim)"
    },
    {
        "image_path": "heatmap.png",
        "title_description": "Density heatmap of variable m across categories C1 to C20 with percentile overlays",
        "chart_type": "density_heatmap",
        "claim": "The density of m at a value of -3 is consistently high (above 0.4) across all categories.",
        "truth" : "(REFUTED claim)"
    }

]

In [8]:
output_json_path = "basemodel_results_box_heat_plot.json"
simulation_results = []

print("🚀 Starting Continuous Generative Verification Pipeline...\n" + "="*60)

for idx, sample in enumerate(simulation_samples):
    print(f"\n--- Processing Sample {idx + 1} ---")
    if not os.path.exists(sample["image_path"]):
        print(f"⚠️  Image not found: {sample['image_path']}. Skipping.")
        continue

    description = sample["title_description"]

    # ==========================================
    # SINGLE PASS: IMAGE → VERDICT + RATIONALE
    # ==========================================
    prompt_text = f"Claim: {sample['claim']}"
    if description:
            prompt_text = f"Chart Description: {description}\n{prompt_text}"

    ver_messages = [
        {
            "role": "system",
            "content": (
                "You are an expert chart fact-checker. "
                "You will be given a chart image, its description, and a claim about it. "
                "Carefully read the chart values and respond in this exact format:\n"
                "Rationale: <one concise sentence citing the specific chart values>\n"
                "Verdict: <SUPPORTED or REFUTED>"
            )
        },
        {
            "role": "user",
            "content": [
                {"type": "image", "image": sample["image_path"], "max_pixels": 768 * 768},
                {"type": "text",  "text": prompt_text}
            ]
        }
    ]

    ver_text = qwen_processor.apply_chat_template(
        ver_messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(ver_messages)
    ver_inputs = qwen_processor(
        text=[ver_text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        ver_ids = qwen_model.generate(
            **ver_inputs,
            max_new_tokens=150,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=qwen_processor.tokenizer.pad_token_id,
        )

    ver_trimmed  = ver_ids[0][len(ver_inputs.input_ids[0]):]
    output_text  = qwen_processor.decode(ver_trimmed, skip_special_tokens=True).strip()

    # ── Parse verdict ──────────────────────────────────────────────────────
    pred_val_str = "REFUTED"
    match = re.search(r"Verdict:\s*(SUPPORTED|REFUTED)", output_text, re.IGNORECASE)
    if match:
        pred_val_str = match.group(1).upper()
    elif "supported" in output_text.lower():
        pred_val_str = "SUPPORTED"

    # ── Parse rationale (everything before the Verdict line) ──────────────
    if match:
        pred_rationale = output_text[:match.start()].replace("Rationale:", "").strip()
    else:
        pred_rationale = output_text

    # ── Console output ─────────────────────────────────────────────────────
    print(f"Claim           : {sample['claim']}")
    print(f"Truth           : {sample['truth'].strip()}")
    print(f"Prediction      : {pred_val_str}")
    print(f"Rationale       : {pred_rationale}")

    # ── Incremental save ───────────────────────────────────────────────────
    simulation_results.append({
        "sample_id":   idx + 1,
        "image_path":  sample["image_path"],
        "chart_type":  sample["chart_type"],
        "claim":       sample["claim"],
        "truth":       sample["truth"].strip(),
        "prediction":  pred_val_str,
        "rationale":   pred_rationale,
    })

    with open(output_json_path, "w", encoding="utf-8") as f:
        json.dump(simulation_results, f, indent=4, ensure_ascii=False)

print("\n" + "="*60 + "\n✅ Simulation Complete!")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


🚀 Starting Continuous Generative Verification Pipeline...

--- Processing Sample 1 ---
Claim           : In Experiment 18, Data Set 2 has a higher median value than Data Set 1.
Truth           : (SUPPORTED claim)
Prediction      : SUPPORTED
Rationale       : The median for Data Set 2 is above that of Data Set 1 in Experiment 18.

--- Processing Sample 2 ---
Claim           : Across most experiments shown, (blue) consistently displays a larger number of lower-end outliers compared to Data Set 2.
Truth           : (SUPPORTED claim)
Prediction      : SUPPORTED
Rationale       : The box plots for Data Set 1 (blue) generally have longer lower whiskers indicating more variability at the lower end compared to Data Set 2 (red).

--- Processing Sample 3 ---
Claim           : In Experiment 24, Data Set 2 shows a significantly higher maximum value, including outliers, than Data Set 1.
Truth           : (REFUTED claim)
Prediction      : SUPPORTED
Rationale       : The box plot for Data Set 2 in Ex